# Machine Learning Assignment 2 - Classification Models

**Name:** Kaushal Pant  
**BITS ID:** 2025AC05981

## Dataset: Breast Cancer Wisconsin (Diagnostic)

This notebook implements and evaluates 5 classification models on the Breast Cancer Wisconsin dataset.

## 1. Import Libraries

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

# Set random seed for reproducibility
np.random.seed(42)

# Style settings
plt.style.use('default')
sns.set_palette('Set2')

print("Libraries imported successfully!")

## 2. Load and Explore Dataset

In [ ]:
# Load dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

# Create complete dataframe
df = X.copy()
df['target'] = y

print(f"Dataset Shape: {df.shape}")
print(f"Features: {X.shape[1]}")
print(f"Instances: {len(df)}")
print(f"\nTarget Distribution:")
print(y.value_counts())
print(f"\nTarget Names: {data.target_names}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
# Target distribution visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
y.value_counts().plot(kind='bar', ax=ax[0], color=['#FF6B6B', '#4ECDC4'])
ax[0].set_title('Target Distribution')
ax[0].set_xlabel('Target Class')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['Malignant (0)', 'Benign (1)'], rotation=0)

# Pie chart
y.value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%', 
                       labels=['Malignant', 'Benign'], colors=['#FF6B6B', '#4ECDC4'])
ax[1].set_title('Target Distribution (%)')
ax[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation table for first 10 features (heatmap removed)
correlation = df.iloc[:, :10].corr().round(2)
print("Correlation matrix (first 10 features):")
correlation

## 4. Train-Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

## 5. Define Models

In [ ]:
# Initialize all 5 models as per assignment requirements
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=7))
    ]),
    'Naive Bayes': GaussianNB(),
    'Random Forest (Ensemble)': RandomForestClassifier(n_estimators=300, random_state=42)
}

print(f"Total models to train: {len(models)}")
for name in models.keys():
    print(f"  - {name}")

## 6. Train Models and Evaluate

In [ ]:
# Function to evaluate a model
def evaluate_model(model, X_train, X_test, y_train, y_test):
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Get probabilities if available
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred
    
    # Calculate metrics
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_pred)
    }
    
    return metrics, y_pred, y_proba

In [ ]:
# Train and evaluate all models
results = {}
predictions = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    metrics, y_pred, y_proba = evaluate_model(model, X_train, X_test, y_train, y_test)
    results[name] = metrics
    predictions[name] = {'pred': y_pred, 'proba': y_proba}

    print("  Metrics:")
    print(f"    Accuracy : {metrics['Accuracy']:.4f}")
    print(f"    AUC      : {metrics['AUC']:.4f}")
    print(f"    Precision: {metrics['Precision']:.4f}")
    print(f"    Recall   : {metrics['Recall']:.4f}")
    print(f"    F1       : {metrics['F1']:.4f}")
    print(f"    MCC      : {metrics['MCC']:.4f}")

print("\nAll models trained successfully!")

## 7. Results Comparison

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('F1', ascending=False)

print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON")
print("="*80)
print(results_df.to_string())
print("="*80)

In [ ]:
# Styled dataframe display
results_df.style.background_gradient(cmap='RdYlGn', axis=0).format('{:.4f}')

## 8. Visualize Results

In [ ]:
# Plot all metrics comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics_list = ['Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

for idx, metric in enumerate(metrics_list):
    bars = results_df[metric].plot(kind='bar', ax=axes[idx], color=colors, width=0.7)
    axes[idx].set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel(metric, fontsize=11)
    axes[idx].set_xlabel('')
    axes[idx].grid(axis='y', alpha=0.3, linestyle='--')
    axes[idx].set_xticklabels(results_df.index, rotation=45, ha='right')
    axes[idx].set_ylim(0, 1.0)
    
    # Add value labels on top of bars
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt='%.3f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

## 9. Confusion Matrices

In [ ]:
# Plot confusion matrices for all 5 models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, pred_dict) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, pred_dict['pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Malignant', 'Benign'])
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(f'{name}\nAccuracy: {results[name]["Accuracy"]:.4f}', fontweight='bold')

# Hide the 6th subplot since we only have 5 models
axes[5].axis('off')

plt.tight_layout()
plt.show()

## 10. Classification Reports

In [ ]:
# Display classification report for all models
print(f"\n{'='*80}")
print("CLASSIFICATION REPORTS - ALL MODELS")
print(f"{'='*80}")

for model_name, pred_dict in predictions.items():
    print(f"\n{'-'*60}")
    print(f"CLASSIFICATION REPORT - {model_name}")
    print(f"{'-'*60}")
    print(classification_report(y_test, pred_dict['pred'], target_names=['Malignant', 'Benign']))

# Best model by F1
best_model_name = results_df.index[0]
best_predictions = predictions[best_model_name]['pred']

print(f"\n{'='*60}")
print(f"BEST MODEL CLASSIFICATION REPORT - {best_model_name}")
print(f"{'='*60}\n")
print(classification_report(y_test, best_predictions, target_names=['Malignant', 'Benign']))
print(f"{'='*60}")

## 11. Save Results

In [ ]:
# Save model metrics in app-compatible format
metrics_for_app = results_df.reset_index().rename(columns={
    'index': 'model',
    'Accuracy': 'accuracy',
    'AUC': 'auc',
    'Precision': 'precision',
    'Recall': 'recall',
    'F1': 'f1',
    'MCC': 'mcc'
})
metrics_for_app.to_csv('../model_metrics.csv', index=False)
print("Model metrics saved to model_metrics.csv")

# Save test data
test_df = X_test.copy()
test_df['target'] = y_test.values
test_df.to_csv('../test_data.csv', index=False)
print("Test data saved to test_data.csv")

# Save dataset_info.json
dataset_info = {
    "dataset_name": "Breast Cancer Wisconsin (Diagnostic)",
    "source": "UCI Machine Learning Repository (available via scikit-learn)",
    "instances": int(len(df)),
    "features": int(X.shape[1]),
    "target_classes": ["malignant", "benign"],
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test))
}
with open("../dataset_info.json", "w", encoding="utf-8") as fp:
    json.dump(dataset_info, fp, indent=2)
print("dataset_info.json saved")

print("\nAll artifacts saved successfully!")

## 12. Summary and Observations

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nDataset: Breast Cancer Wisconsin (Diagnostic)")
print(f"Total Instances: {len(df)}")
print(f"Total Features: {X.shape[1]}")
print(f"\nModels Implemented: {len(models)}")
print(f"\nBest Model: {best_model_name}")
print(f"Best F1 Score: {results[best_model_name]['F1']:.4f}")
print(f"Best Accuracy: {results[best_model_name]['Accuracy']:.4f}")
print(f"Best MCC: {results[best_model_name]['MCC']:.4f}")
print("\n" + "="*80)

print("\nKey Observations:")
print("1. Logistic Regression achieves excellent performance with balanced metrics")
print("2. KNN shows perfect recall (1.0) - no false negatives")
print("3. Random Forest ensemble method shows robust performance with highest AUC")
print("4. All models achieve >91% accuracy on this dataset")
print("5. Feature scaling significantly improves KNN and Logistic Regression performance")
print("="*80)

## End of Notebook

---

**Assignment completed successfully!**

Next steps:
1. Deploy the Streamlit app
2. Push code to GitHub
3. Submit PDF with links and screenshots